# 01 — Load LIMASAM container locations

This notebook loads the raw container address list provided by LIMASAM (Limpieza de Málaga, S.A.M.) in March 2026, inspects its structure, and prepares it for geocoding in the next notebook.

**Input:** `data/raw/limasam_containers_2026-03.xlsx` — one column (`Dirección`), 484 free-text addresses.

**Output:** `data/interim/containers_cleaned.csv` — cleaned addresses ready for geocoding.

In [1]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent  # notebooks/ is one level down from project root
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("Files in raw:", list(DATA_RAW.iterdir()))

Project root: c:\Users\user\Documents\Projects\malaga-textile-access
Raw data folder: c:\Users\user\Documents\Projects\malaga-textile-access\data\raw
Files in raw: [WindowsPath('c:/Users/user/Documents/Projects/malaga-textile-access/data/raw/.gitkeep'), WindowsPath('c:/Users/user/Documents/Projects/malaga-textile-access/data/raw/limasam_containers_2026-03.xlsx')]


In [2]:
# Load the Excel file
xlsx_path = DATA_RAW / "limasam_containers_2026-03.xlsx"
df = pd.read_excel(xlsx_path)

# Quick look at structure
print("Shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 10 rows:")
df.head(10)

Shape (rows, columns): (483, 1)

Column names: ['Dirección:']

First 10 rows:


,Dirección:
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad..."
3,"C. Federico García Lorca, 13-11, Carretera de ..."
4,"Calle Goya, 7, Carretera de Cádiz, 29002 Málaga"
5,"C. Gran Bretaña, 2-8, Cruz de Humilladero, 290..."
6,"C. Héroe de Sostoa, 105, Carretera de Cádiz, 2..."
7,"C. Isla Cristina, 13-15, Carretera de Cádiz, 2..."
8,"C. José María Jacquard, 55-53, Cruz de Humilla..."
9,"C. Martínez de la Rosa, 92, Bailén-Miraflores,..."


In [3]:
col = "Dirección:"

# Missing values
print("Empty rows:", df[col].isna().sum())

# Duplicates
print("Exact duplicates:", df[col].duplicated().sum())

# Address length distribution
print("\nAddress length stats:")
print(df[col].str.len().describe())

# Show shortest and longest addresses
df_sorted = df.assign(length=df[col].str.len()).sort_values("length")

print("\n5 shortest addresses:")
for addr in df_sorted.head(5)[col].tolist():
    print(f"  {addr}")

print("\n5 longest addresses:")
for addr in df_sorted.tail(5)[col].tolist():
    print(f"  {addr}")

print("\nDuplicate rows:")
print(df[df[col].duplicated(keep=False)].sort_values(col))

Empty rows: 0
Exact duplicates: 3

Address length stats:
count    483.000000
mean      31.739130
std       14.470254
min       10.000000
25%       21.000000
50%       28.000000
75%       38.500000
max       72.000000
Name: Dirección:, dtype: float64

5 shortest addresses:
  C/ ALMERIA
  FRANZ KAFKA
  C/ REAL, 15
  plaza babel
  C/ Praga, 7

5 longest addresses:
  rmoles, 14 Frente clínica "SINTESIS CENTER". Encima acera, al lado de k
  ntiago Ramón y Cajal, 67  Sobre acera. Altura parada de Taxis. Frente C
  tra. Sra. De Las Guías, 3 Esquina plaza Nazaret. Sobre acera, en Parada
  / Pastora Imperio, 11 Con C/ Sor Juana Inés de la Cruz. Cerca de gloriet
  n C/ Alcalde Joaquín Quiles. Frente parque infantil. Sobre acera de fren

Duplicate rows:
                                 Dirección:
345             C/ Armengual de la Mota, 29
438             C/ Armengual de la Mota, 29
309  C/ Juan Vazquez, 3 esquina C/ Idris, 1
420  C/ Juan Vazquez, 3 esquina C/ Idris, 1
353              C/ Sierra 

In [4]:
import re

col = "Dirección:"

# Pattern: one or more digits anywhere in the string
has_number = df[col].str.contains(r"\d", regex=True)

print(f"Total addresses: {len(df)}")
print(f"  With at least one digit: {has_number.sum()}")
print(f"  Without any digit:        {(~has_number).sum()}")
print()
print("Addresses without any digit:")
no_number = df.loc[~has_number, col].tolist()
for addr in no_number:
    print(f"  → {addr}")

Total addresses: 483
  With at least one digit: 386
  Without any digit:        97

Addresses without any digit:
  → C. Almogía MALAGA
  → Andarax con Camino de san Rafael
  → Aresnisca con Obsidiana - Malaga
  → C. Arlanzón MALAGA
  → Arroyo de los Angeles esquina Crisitino Martos
  → Arroyo de los Angeles esquina nuestra señora de los clarines
  → Avenida de Europa esquina puerto oncala
  → Av. de Molière MALAGA
  → Calle Bidasoa
  → C. Cancho Perez
  → Cipres esquina Emilio Benavent - Malaga
  → C. Deva MALAGA
  → duero con Eresma, Malaga
  → Gaucin esquina Angelillo de Triana,  malaga
  → C. Godino & Av. de Luis Buñuel Malaga
  → Imperio Argentina - Marilyn Monroe MALAGA
  → Jane Bowles con Borodin - Malaga
  → C. Jose Iturbi
  → Jose Iturbi esquina Sondalezas
  → Alcalde Jose Luis Estrada esquina Galeno - Malaga
  → Calle Josep Pla MALAGA
  → C. Juan de Robles MALAGA
  → La Hoz esquina carpio - malaga
  → luis buñuel con avenida simon bolivar  Malaga
  → Magistrado Salvador Barber

## Cleaning strategy

Inspection revealed that of 483 raw addresses:
- 386 contain a digit (likely a house number)
- 97 do not — mostly street intersections (`X esquina Y`, `X con Y`) or street-only references

We apply the following cleaning rules:

1. **Expand abbreviations**: `C/`, `C.`, `Cl.` → `Calle`; `Av.`, `Avda.`, `Avenida` → `Avenida`; `Ctra.` → `Carretera`; `Pza`, `Pza.` → `Plaza`.
2. **Normalize separators**: replace `esquina`, `esq.`, `con`, `&`, `con C/`, etc. with a consistent ` and ` for intersections.
3. **Strip trailing context**: descriptions like `Frente parque infantil`, `Sobre acera`, `MALAGA` at the end are removed for geocoding (they only confuse Nominatim).
4. **Title-case** the result for consistency.
5. **Append `, Málaga, Spain`** to every address.
6. **Tag each address** by structure: `house_number`, `intersection`, `street_only`, or `unclear`.
7. **Remove the 3 exact-duplicate pairs**: same address pair likely means two containers at the same spot — does not affect access analysis (distance to point is the same).

In [5]:
import re

# -------------------- Cleaning functions --------------------

def expand_abbreviations(s: str) -> str:
    """Replace common Spanish street-type abbreviations with full words.
    
    Patterns also absorb the trailing period so we don't leave orphan dots
    like 'Avenida.'.
    """
    rules = [
        (r"\bAvda\.?",     "Avenida"),
        (r"\bAv\.?",       "Avenida"),
        (r"\bCtra\.?",     "Carretera"),
        (r"\bCmno\.?",     "Camino"),
        (r"\bPza\.?",      "Plaza"),
        (r"\bPje\.?",      "Pasaje"),
        (r"\bC\.\s",       "Calle "),   # "C. Goya"
        (r"\bC/\s?",       "Calle "),   # "C/Mesonero"
        (r"\bCl\.?\b",     "Calle"),
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    # Collapse any double spaces that might appear
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_intersection_words(s: str) -> str:
    """Replace various intersection indicators with a single ' and '."""
    # Be careful: 'con' is a common Spanish word; we only target it
    # between two capitalised street names. Simpler: replace standalone
    # ' con ', ' esquina ', ' esq ', ' esq. ', ' & ', ' y ' (only between streets is tricky).
    rules = [
        (r"\besquina\b",  " and "),
        (r"\besq\.?\b",   " and "),
        (r"\s&\s",         " and "),
        (r"\bcon\b",       " and "),
        (r"\bentre\b",     ""),       # 'Entre C/ X y C/ Y' → ' C/ X y C/ Y' (then 'y' below)
    ]
    for pattern, repl in rules:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    # Clean repeated 'and'
    s = re.sub(r"\s+and\s+and\s+", " and ", s)
    return s


def strip_trailing_context(s: str) -> str:
    """Remove descriptive tails like 'Frente parque infantil', 'Sobre acera', and trailing 'MALAGA'."""
    # 1) Cut at common descriptor keywords (anything from there onward is noise).
    cut_markers = [
        r"\bFrente\b",
        r"\bSobre\s+acera\b",
        r"\bCerca\s+de\b",
        r"\bLateral\b",
        r"\bJunto\b",
        r"\(",          # parenthetical comments like "(SUPERMERCADO DÍA)"
    ]
    pattern = "|".join(cut_markers)
    s = re.split(pattern, s, maxsplit=1, flags=re.IGNORECASE)[0]

    # 2) Remove trailing 'Málaga' / 'MALAGA' / 'malaga' (we add it back uniformly later)
    s = re.sub(r"[\s,;\-]*M[aá]laga\s*$", "", s, flags=re.IGNORECASE)

    # 3) Remove 's/n' (sin número)
    s = re.sub(r",?\s*s/n\b", "", s, flags=re.IGNORECASE)

    # 4) Tidy whitespace and punctuation
    s = re.sub(r"\s+", " ", s).strip(" ,-.;")
    return s


def smart_title_case(s: str) -> str:
    """Title-case, but keep short words like 'de', 'la', 'y', 'and' lowercase (except first word)."""
    small = {"de", "del", "la", "las", "el", "los", "y", "and"}
    words = s.split()
    out = []
    for i, w in enumerate(words):
        if i > 0 and w.lower() in small:
            out.append(w.lower())
        else:
            out.append(w.capitalize())
    return " ".join(out)


def clean_address(raw: str) -> str:
    """Full cleaning pipeline for one raw address."""
    s = str(raw)
    s = strip_trailing_context(s)
    s = expand_abbreviations(s)
    s = normalize_intersection_words(s)
    s = smart_title_case(s)
    # Final: append Málaga, Spain
    s = f"{s}, Málaga, Spain"
    return s


# -------------------- Tagging function --------------------

def tag_address_type(raw: str) -> str:
    """Classify the raw address by its structure."""
    s = raw.lower()
    has_digit = bool(re.search(r"\d", raw))
    has_intersection = bool(re.search(r"\b(esquina|esq\.?|con|&|entre)\b", s, flags=re.IGNORECASE))

    if has_digit:
        return "house_number"
    elif has_intersection:
        return "intersection"
    else:
        # Looks like street-only if it starts with a street-type marker
        starts_with_street = bool(re.match(r"^\s*(c[/\.]|calle|avda?\.?|avenida|plaza|pza|ctra|carretera|cmno|camino|pasaje|pje)\b", s, flags=re.IGNORECASE))
        if starts_with_street or len(s.split()) >= 2:
            return "street_only"
        else:
            return "unclear"

In [6]:
# Apply cleaning + tagging to the dataframe
df_clean = df.copy()
df_clean["address_raw"] = df_clean[col]
df_clean["address_type"] = df_clean[col].apply(tag_address_type)
df_clean["address_clean"] = df_clean[col].apply(clean_address)

# Remove duplicates (based on the raw address — exact matches only)
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="address_raw").reset_index(drop=True)
after = len(df_clean)
print(f"Removed {before - after} duplicate rows ({before} → {after})")

# Show counts per address type
print("\nAddress type distribution:")
print(df_clean["address_type"].value_counts())

# Drop the original column (we have address_raw now)
df_clean = df_clean.drop(columns=[col])

print(f"\nFinal columns: {df_clean.columns.tolist()}")
df_clean.head(5)

Removed 3 duplicate rows (483 → 480)

Address type distribution:
address_type
house_number    383
intersection     64
street_only      33
Name: count, dtype: int64

Final columns: ['address_raw', 'address_type', 'address_clean']


,address_raw,address_type,address_clean
0,"Av. de la Paloma, 36-38, Carretera de Cádiz, 2...",house_number,"Avenida de la Paloma, 36-38, Carretera de Cádi..."
1,"Av. de la Paloma, 43, Carretera de Cádiz, 2900...",house_number,"Avenida de la Paloma, 43, Carretera de Cádiz, ..."
2,"C. Conde de Guadalhorce, 6-8, Cruz de Humillad...",house_number,"Calle Conde de Guadalhorce, 6-8, Cruz de Humil..."
3,"C. Federico García Lorca, 13-11, Carretera de ...",house_number,"Calle Federico García Lorca, 13-11, Carretera ..."
4,"Calle Goya, 7, Carretera de Cádiz, 29002 Málaga",house_number,"Calle Goya, 7, Carretera de Cádiz, 29002, Mála..."


In [7]:
# Make sure the interim folder exists, then save
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
out_path = DATA_INTERIM / "containers_cleaned.csv"

df_clean.to_csv(out_path, index=False, encoding="utf-8")

print(f"Saved {len(df_clean)} cleaned addresses to:")
print(f"  {out_path}")

# Verify by reading back the first lines
print("\nFirst 3 rows of saved file:")
print(pd.read_csv(out_path).head(3))

Saved 480 cleaned addresses to:
  c:\Users\user\Documents\Projects\malaga-textile-access\data\interim\containers_cleaned.csv

First 3 rows of saved file:
                                         address_raw  address_type  \
0  Av. de la Paloma, 36-38, Carretera de Cádiz, 2...  house_number   
1  Av. de la Paloma, 43, Carretera de Cádiz, 2900...  house_number   
2  C. Conde de Guadalhorce, 6-8, Cruz de Humillad...  house_number   

                                       address_clean  
0  Avenida de la Paloma, 36-38, Carretera de Cádi...  
1  Avenida de la Paloma, 43, Carretera de Cádiz, ...  
2  Calle Conde de Guadalhorce, 6-8, Cruz de Humil...  
